## 1. Confirm the GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Notebook settings (right panel) -> Accelerator -> GPU T4 x2'

## 2. Locate the attached dataset

Auto-detects the mounted dataset by looking for `Dataset/Img` under `/kaggle/input`, so you don't need to hardcode your dataset's slug.

In [ ]:
import os

def find_input_root(target=os.path.join('Dataset', 'Img')):
    for root, dirs, files in os.walk('/kaggle/input'):
        if root.endswith(target):
            return os.path.dirname(os.path.dirname(root))  # strip 'Img', then 'Dataset'
    return None

INPUT_ROOT = find_input_root()
assert INPUT_ROOT, (
    'No Dataset/Img found anywhere under /kaggle/input. '
    'Run print(os.listdir("/kaggle/input/datasets")) to inspect the real layout.'
)
print('Dataset mount:', INPUT_ROOT)
print('memes    :', len(os.listdir(f'{INPUT_ROOT}/Dataset/Img')))
print(sorted(os.listdir(INPUT_ROOT)))

## 3. Install dependencies

Kaggle notebooks already ship a CUDA build of torch, so only the extras are installed — notably CLIP from source, exactly as the original `requirements.txt` specifies.

In [ ]:
!pip install -q transformers sentencepiece imbalanced-learn madgrad ftfy regex
!pip install -q git+https://github.com/openai/CLIP.git

import clip, transformers
print('transformers', transformers.__version__)
print('clip OK')

## 4. Set up a writable working copy

`/kaggle/input` is read-only, but `main.py` needs to create `Saved_Models/`, `Outputs/` and a `.cache/` next to `Scripts/`. So `Scripts/` (a few hundred KB) is copied to `/kaggle/working`; `Dataset/` (365 MB of images) stays on the read-only input mount and is referenced by an absolute path — no need to duplicate it.

In [ ]:
import shutil

ROOT = '/kaggle/working/Replication'
os.makedirs(ROOT, exist_ok=True)

if os.path.isdir(f'{ROOT}/Scripts'):
    shutil.rmtree(f'{ROOT}/Scripts')
shutil.copytree(f'{INPUT_ROOT}/Scripts', f'{ROOT}/Scripts')
shutil.copy(f'{INPUT_ROOT}/requirements.txt', ROOT)

DATASET_PATH = f'{INPUT_ROOT}/Dataset'  # absolute path -> main.py reads straight from the input mount
print('Working scripts at:', f'{ROOT}/Scripts')
print('Reading dataset from (read-only):', DATASET_PATH)

## 5. Splits summary (cf. paper Table 1)

In [ ]:
import pandas as pd
for name in ['training_set', 'validation_set', 'testing_set']:
    df = pd.read_csv(f'{DATASET_PATH}/{name}.csv')
    print(f'{name:<16} {len(df):>5} rows   ', dict(df["Label"].value_counts()))

## 6. Smoke test

Runs the full pipeline on a handful of memes for one epoch. This is **not a result** — it only proves the data, the model and the metrics all wire up before committing to a long run.

In [ ]:
%cd {ROOT}/Scripts
!python main.py --dataset "{DATASET_PATH}" --subset 24 --n_iter 1 --run_name kaggle_smoketest

## 7. The real run

Paper hyperparameters (Appendix A): batch 4, 20 epochs, lr 5e-5, 16 attention heads, max_len 70.

The best-validation-accuracy checkpoint is kept and used for the test evaluation, as in the original. Mind Kaggle's session time limit (~9–12 hrs) — this run is expected to take 1–2 hrs on a T4.

In [ ]:
%cd {ROOT}/Scripts
!python main.py --dataset "{DATASET_PATH}" --run_name maf_full --batch_size 16 --n_iter 5 --lrate 5e-5 --heads 16 --max_len 70

## 8. Ablations and variants (optional)

- `--attn_variant paper` — the attention operand order the **paper text** describes (Q from text, K/V from vision), rather than the order the **released code** implements. See README §5.4.
- `--fix_scheduler` — steps the LR scheduler per batch instead of per epoch, correcting the original's scheduler bug. See README §5.5.

In [ ]:
%cd {ROOT}/Scripts
!python main.py --dataset "{DATASET_PATH}" --run_name maf_paper_attn --attn_variant paper --batch_size 16 --n_iter 5 --lrate 5e-5 --heads 16 --max_len 70
!python main.py --dataset "{DATASET_PATH}" --run_name maf_fixed_sched --fix_scheduler --batch_size 16 --n_iter 5 --lrate 5e-5 --heads 16 --max_len 70

## 9. Results

Compares every run in `Outputs/` against the paper's published MAF row.

**These numbers are not directly comparable to the paper's** — our task is 4-way rather than 5-way, and our captions are raw OCR (optionally denoised) with no manual correction pass. See README §5.

In [ ]:
import glob, json
import pandas as pd

rows = []
for path in sorted(glob.glob(f'{ROOT}/Outputs/results_*.json')):
    r = json.load(open(path, encoding='utf-8'))
    rows.append({
        'run': r['run_name'],
        'Acc': round(r['accuracy'], 3),
        'WF1': round(r['weighted_f1'], 3),
        'MacroF1': round(r['macro_f1'], 3),
        'MMAE': round(r['mmae'], 3),
    })
rows.append({'run': 'PAPER MAF (5-way MIMOSA)', 'Acc': 0.741, 'WF1': 0.742, 'MacroF1': None, 'MMAE': 0.645})
print(pd.DataFrame(rows).to_string(index=False))

## 10. Confusion matrix

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns

RUN = 'maf_full'
r = json.load(open(f'{ROOT}/Outputs/results_{RUN}.json', encoding='utf-8'))

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(r['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            xticklabels=r['target_names'], yticklabels=r['target_names'], cbar=False)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'MAF — {RUN}')
plt.tight_layout(); plt.show()

## 11. Results persist automatically

Everything under `/kaggle/working` (including `{ROOT}/Outputs` and `{ROOT}/Saved_Models`) is kept as the notebook's **Output** when you commit/save the session — download it from the notebook's "Output" tab afterward. No extra copy step needed (unlike Colab, which needs an explicit copy back to Drive).

In [ ]:
!ls -lh {ROOT}/Outputs {ROOT}/Saved_Models